In [1]:
import pandas as pd

In [2]:
df = pd.read_csv('/content/100_Unique_QA_Dataset.csv')

df.head()

,question,answer
0,What is the capital of France?,Paris
1,What is the capital of Germany?,Berlin
2,Who wrote 'To Kill a Mockingbird'?,Harper-Lee
3,What is the largest planet in our solar system?,Jupiter
4,What is the boiling point of water in Celsius?,100


In [3]:
# tokenize
def tokenize(text):
  text = text.lower()
  text = text.replace('?','')
  text = text.replace("'","")
  return text.split()

In [4]:
tokenize('What is the capital of France?')

['what', 'is', 'the', 'capital', 'of', 'france']

In [5]:
# vocab
vocab = {'<UNK>':0} #dictionaryll

In [6]:
def build_vocab(row):
  tokenized_question = tokenize(row['question'])
  tokenized_answer = tokenize(row['answer'])

  merged_tokens = tokenized_question + tokenized_answer

  for token in merged_tokens:

    if token not in vocab:
      vocab[token] = len(vocab) #current length of vocab


In [7]:
df.apply(build_vocab, axis=1)

,0
0,None
1,None
2,None
3,None
4,None
...,...
85,None
86,None
87,None
88,None


In [8]:
len(vocab)

324

In [10]:
#convert words to numerical indices

def text_to_indices(text,vocab):

  indexed_text = []

  for token in tokenize(text):
    if token in vocab:
      indexed_text.append(vocab[token])
    else:
      indexed_text.append(vocab['<UNK>'])

  return indexed_text



In [11]:
text_to_indices("What is campusx",vocab)

[1, 2, 0]

In [12]:
import torch
from torch.utils.data import Dataset,DataLoader

In [14]:
class QADataset(Dataset):

  def __init__(self,df,vocab):
    self.df = df
    self.vocab = vocab

  def __len__(self):
    return len(self.df)

  def __getitem__(self,idx):

    numerical_question=text_to_indices(self.df.iloc[idx]['question'],self.vocab)
    numerical_answer=text_to_indices(self.df.iloc[idx]['answer'],self.vocab)

    return torch.tensor(numerical_question),torch.tensor(numerical_answer)

In [15]:
dataset = QADataset(df, vocab)

In [16]:
dataloader = DataLoader(dataset, batch_size=1, shuffle=True)

In [19]:
for question, answer in dataloader:
  print(question, answer[0])

tensor([[78, 79, 80, 81, 82, 83, 84]]) tensor([85])
tensor([[  1,   2,   3, 146,  86,  19, 192, 193]]) tensor([194])
tensor([[  1,   2,   3,   4,   5, 279]]) tensor([280])
tensor([[ 42, 255,   2, 256,  83, 257, 258]]) tensor([259])
tensor([[ 10,  11, 157, 158, 159]]) tensor([160])
tensor([[ 42, 174,   2,  62,  39, 175, 176,  12, 177, 178]]) tensor([179])
tensor([[10,  2,  3, 66,  5, 67]]) tensor([68])
tensor([[ 42, 137,   2, 226,  12,   3, 227, 228]]) tensor([155])
tensor([[ 42, 216, 118, 217, 218,  19,  14, 219,  43]]) tensor([220])
tensor([[10, 75, 76]]) tensor([77])
tensor([[10, 96,  3, 97]]) tensor([98])
tensor([[  1,   2,   3, 163, 164, 165,  83,  84]]) tensor([166])
tensor([[ 42,  18, 118,   3, 186, 187]]) tensor([188])
tensor([[ 78,  79, 129,  81,  19,   3,  21,  22]]) tensor([36])
tensor([[ 10, 308,   3, 309, 310]]) tensor([311])
tensor([[ 1,  2,  3,  4,  5, 99]]) tensor([100])
tensor([[ 1,  2,  3,  4,  5, 53]]) tensor([54])
tensor([[ 78,  79, 288,  81,  19,  14, 289]]) tensor(

In [20]:
import torch.nn as nn

In [21]:
class SimpleRNN(nn.Module):

  def __init__(self, vocab_size):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, embedding_dim=50)
    self.rnn = nn.RNN(50, 64, batch_first=True)
    self.fc = nn.Linear(64, vocab_size)

  def forward(self, question):
    embedded_question = self.embedding(question)
    hidden, final = self.rnn(embedded_question)
    output = self.fc(final.squeeze(0))

    return output

In [22]:
x = nn.Embedding(324, embedding_dim=50)
y = nn.RNN(50, 64, batch_first=True)
z = nn.Linear(64, 324)

a = dataset[0][0].reshape(1,6)
print("shape of a:", a.shape)
b = x(a)
print("shape of b:", b.shape)
c, d = y(b)
print("shape of c:", c.shape)
print("shape of d:", d.shape)

e = z(d.squeeze(0))

print("shape of e:", e.shape)

shape of a: torch.Size([1, 6])
shape of b: torch.Size([1, 6, 50])
shape of c: torch.Size([1, 6, 64])
shape of d: torch.Size([1, 1, 64])
shape of e: torch.Size([1, 324])


In [23]:
learning_rate = 0.001
epochs = 20

In [24]:
model = SimpleRNN(len(vocab))

In [25]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

In [26]:
# training loop

for epoch in range(epochs):

  total_loss = 0

  for question, answer in dataloader:

    optimizer.zero_grad()

    # forward pass
    output = model(question)

    # loss -> output shape (1,324) - (1)
    loss = criterion(output, answer[0])

    # gradients
    loss.backward()

    # update
    optimizer.step()

    total_loss = total_loss + loss.item()

  print(f"Epoch: {epoch+1}, Loss: {total_loss:4f}")

Epoch: 1, Loss: 525.207570
Epoch: 2, Loss: 451.202899
Epoch: 3, Loss: 373.280187
Epoch: 4, Loss: 315.455479
Epoch: 5, Loss: 265.954813
Epoch: 6, Loss: 218.883982
Epoch: 7, Loss: 175.423016
Epoch: 8, Loss: 137.143831
Epoch: 9, Loss: 105.516067
Epoch: 10, Loss: 80.977640
Epoch: 11, Loss: 62.165263
Epoch: 12, Loss: 48.604353
Epoch: 13, Loss: 38.732105
Epoch: 14, Loss: 31.558550
Epoch: 15, Loss: 26.076279
Epoch: 16, Loss: 21.773210
Epoch: 17, Loss: 18.554616
Epoch: 18, Loss: 15.839414
Epoch: 19, Loss: 13.729186
Epoch: 20, Loss: 11.927728


In [27]:
def predict(model, question, threshold=0.5):

  # convert question to numbers
  numerical_question = text_to_indices(question, vocab)

  # tensor
  question_tensor = torch.tensor(numerical_question).unsqueeze(0)

  # send to model
  output = model(question_tensor)

  # convert logits to probs
  probs = torch.nn.functional.softmax(output, dim=1)

  # find index of max prob
  value, index = torch.max(probs, dim=1)

  if value < threshold:
    print("I don't know")

  print(list(vocab.keys())[index])

In [28]:
predict(model, "What is the largest planet in our solar system?")

jupiter


In [29]:
list(vocab.keys())[7]

'paris'